In [45]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import re
import sys
sys.path.append("../")

from src.data_cleaning import clean_text, extract_nba_injury_type, normalize_empty_strings
from src.feature_engineering import categorize_injury   
from itertools import combinations

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# 1.Data preparation for my analisys

## NFL injuries

In [47]:

nfl = pd.read_csv("../datasets/raw/nfl_injuries_2009_2021.csv")
nba = pd.read_csv("../datasets/raw/nba_injuries_2010-2020.csv")
nhl = pd.read_csv("../datasets/raw/nhl_injuries.csv")
mlb = pd.read_csv("../datasets/raw/mlb_injury.csv")

In [48]:
nfl.head()

,Unnamed: 0,Player_id,Player,year,team,games_in_season,num_games_missing,num_games_injured,injury_types,earliest_injury,latest_injury
0,all_info,AdamMi98,"Adams,Michael",2009,crd,16,0,3,hamstring,8.0,14.0
1,all_info.1,BankJa00,"Banks,Jason",2009,crd,16,0,0,NaN,NaN,NaN
2,all_info.2,BerrBe99,"Berry,Bertrand",2009,crd,16,0,1,illness,13.0,13.0
3,all_info.3,BoldAn00,"Boldin,Anquan",2009,crd,16,1,6,hamstring ankle,1.0,9.0
4,all_info.4,BranAl99,"Branch,Alan",2009,crd,16,0,1,ankle,13.0,13.0


In [49]:
nfl.info()

<class 'pandas.DataFrame'>
RangeIndex: 17389 entries, 0 to 17388
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         17389 non-null  str    
 1   Player_id          17389 non-null  str    
 2   Player             17389 non-null  str    
 3   year               17389 non-null  int64  
 4   team               17389 non-null  str    
 5   games_in_season    17389 non-null  int64  
 6   num_games_missing  17389 non-null  int64  
 7   num_games_injured  17389 non-null  int64  
 8   injury_types       15910 non-null  str    
 9   earliest_injury    17087 non-null  float64
 10  latest_injury      17087 non-null  float64
dtypes: float64(2), int64(4), str(5)
memory usage: 1.5 MB


In [50]:
nfl.describe()

,year,games_in_season,num_games_missing,num_games_injured,earliest_injury,latest_injury
count,17389.000000,17389.000000,17389.000000,17389.000000,17087.000000,17087.000000
mean,2014.747829,15.723906,4.110817,5.640002,5.813308,12.327969
std,3.644730,1.234834,5.132237,4.898407,4.548916,4.439140
min,2009.000000,9.000000,0.000000,0.000000,1.000000,1.000000
25%,2012.000000,16.000000,0.000000,2.000000,2.000000,9.000000
50%,2015.000000,16.000000,2.000000,4.000000,5.000000,14.000000
75%,2018.000000,16.000000,6.000000,8.000000,9.000000,16.000000
max,2021.000000,16.000000,16.000000,16.000000,16.000000,16.000000


In [51]:
nfl.isnull().sum()

Unnamed: 0              0
Player_id               0
Player                  0
year                    0
team                    0
games_in_season         0
num_games_missing       0
num_games_injured       0
injury_types         1479
earliest_injury       302
latest_injury         302
dtype: int64

In [52]:
nfl.shape

(17389, 11)

In [53]:
print("NFL Dataset Duplicates:", nfl.duplicated().sum())
print("\nUnique values per column:\n", nfl.nunique())

NFL Dataset Duplicates: 0

Unique values per column:
 Unnamed: 0           17389
Player_id             6023
Player                5947
year                    13
team                    32
games_in_season          4
num_games_missing       17
num_games_injured       17
injury_types          3760
earliest_injury         16
latest_injury           16
dtype: int64


In [54]:
nfl.columns.value_counts()

Unnamed: 0           1
Player_id            1
Player               1
year                 1
team                 1
games_in_season      1
num_games_missing    1
num_games_injured    1
injury_types         1
earliest_injury      1
latest_injury        1
Name: count, dtype: int64

**NFL Dataset Observations:**
*   **Description:** This dataset tracks injuries in the National Football League (NFL) over a span of multiple seasons (2009-2021). It includes information on the players, specific games, the type of injury sustained, and the resulting missed playing time.
*   **Data Source:** https://github.com/clementzach/BTS_260_project/blob/master/Data/data_build/injuries_2009_2021.csv
*   **Limitations:** The dataset does not detail multiple separate injury events sequentially occurring within the same match. Certain vague injury categorizations may require domain knowledge to decipher.
*   **Columns:** The dataset consists of 11 columns, including key features such as `injury_types`, `num_games_missing`, `num_games_injured`, and `year`.


**Additional Insights:**
*   **Duplicates:** 0 duplicate rows found.
*   **Unique Values:** High cardinality in `injury_types` (3760 unique values) indicates we might need to group or normalize the strings (e.g., lowercase, stemming) to avoid feature explosion.
* **Injury type column?** `injury_types`
* **Games missed/recovery/duration?** `num_games_missing`, `num_games_injured`
* **How are injuries described?** Described categorically in `injury_types`.
* **Season/year?** `year`
* **Cleanest/most chaotic?** Fairly clean and structured. Size: 17,389 rows, 11 columns.

## NBA injuries

In [55]:
nba.head()

,Date,Team,Acquired,Relinquished,Notes
0,2010-10-03,Bulls,NaN,Carlos Boozer,fractured bone in right pinky finger (out inde...
1,2010-10-06,Pistons,NaN,Jonas Jerebko,torn right Achilles tendon (out indefinitely)
2,2010-10-06,Pistons,NaN,Terrico White,broken fifth metatarsal in right foot (out ind...
3,2010-10-08,Blazers,NaN,Jeff Ayres,torn ACL in right knee (out indefinitely)
4,2010-10-08,Nets,NaN,Troy Murphy,strained lower back (out indefinitely)


In [56]:
nba.info()

<class 'pandas.DataFrame'>
RangeIndex: 27105 entries, 0 to 27104
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Date          27105 non-null  str  
 1   Team          27103 non-null  str  
 2   Acquired      9542 non-null   str  
 3   Relinquished  17560 non-null  str  
 4   Notes         27105 non-null  str  
dtypes: str(5)
memory usage: 1.0 MB


In [57]:
nba.describe()

,Date,Team,Acquired,Relinquished,Notes
count,27105,27103,9542,17560,27105
unique,2103,32,1111,1156,3114
top,2014-04-16,Spurs,Dwyane Wade,Kevin Love,activated from IL
freq,76,1163,54,101,7342


In [58]:
nba.isnull().sum()

Date                0
Team                2
Acquired        17563
Relinquished     9545
Notes               0
dtype: int64

In [59]:
nba.shape

(27105, 5)

In [60]:
print("NBA Dataset Duplicates:", nba.duplicated().sum())
print("\nUnique values per column:\n", nba.nunique())

NBA Dataset Duplicates: 2

Unique values per column:
 Date            2103
Team              32
Acquired        1111
Relinquished    1156
Notes           3114
dtype: int64


In [61]:
nba.columns.value_counts()

Date            1
Team            1
Acquired        1
Relinquished    1
Notes           1
Name: count, dtype: int64

**NBA Dataset Observations:**
*   **Description:** The NBA dataset (2010-2020) tracks injuries and missed games across basketball seasons. Unlike other sports datasets, it usually aggregates injury updates in a free-text "Notes" field rather than standardized anatomical keywords, often gathered through daily injury reports.
*   **Data Source:** (https://www.kaggle.com/datasets/ghopkins/nba-injuries-2010-2018)
*   **Limitations:** Heavily relies on unstandardized free-text logs for injuries and return times. Specific injury body parts are often mixed with generalized statuses like "rest" or "illness".
*   **Columns:** The dataset is very minimal with only 5 columns, relying heavily on `Date` and `Notes`, along with `Team`, `Acquired`, and `Relinquished` to detail team movements related to injuries.



**Additional Insights:**
*   **Duplicates:** 2 duplicate rows found. These should be removed during cleaning.
*   **Unique Values:** `Notes` holds 3,114 unique text entries. Since this is the primary source of injury detail, NLP processing will be necessary to extract meaningful, categorical tags.
* **Injury type column?** No explicit categorical column; described in `Notes`.
* **Games missed/recovery/duration?** Missed games/recovery are written as free-text within the `Notes`.
* **How are injuries described?** Free-text in the `Notes`.
* **Season/year?** Mostly relies on the `Date` column.
* **Cleanest/most chaotic?** Most chaotic/unstructured due to its reliance on free-text note parsing. Size: 27,105 rows, 5 columns.

## NHL Injuries

In [62]:
nhl.head()

,Season,Team,Position,Player,Injury Type,Cap Hit,Chip,Games Missed,Games Missed.1
0,2000/01,Anaheim,F,"Kariya, Paul",Foot,NaN,NaN,16,16
1,2000/01,Anaheim,F,"Leclerc, Mike",Abdominal,NaN,NaN,13,13
2,2000/01,Anaheim,F,"Leclerc, Mike",Knee,NaN,NaN,15,15
3,2000/01,Anaheim,F,"McDonald, Andy",Concussion,NaN,NaN,7,7
4,2000/01,Anaheim,F,"McInnis, Marty",Groin,NaN,NaN,3,3


In [63]:
nhl.info()

<class 'pandas.DataFrame'>
RangeIndex: 24498 entries, 0 to 24497
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Season          24498 non-null  str    
 1   Team            24498 non-null  str    
 2   Position        24498 non-null  str    
 3   Player          24498 non-null  str    
 4   Injury Type     24498 non-null  str    
 5   Cap Hit         17666 non-null  float64
 6   Chip            17666 non-null  float64
 7   Games Missed    24498 non-null  int64  
 8   Games Missed.1  24498 non-null  int64  
dtypes: float64(2), int64(2), str(5)
memory usage: 1.7 MB


In [64]:
nhl.describe()

,Cap Hit,Chip,Games Missed,Games Missed.1
count,17666.000000,17666.000000,24498.000000,24498.000000
mean,2.966112,307.173656,8.145400,8.183566
std,2.239837,671.117390,12.125458,12.122303
min,0.300000,5.182927,1.000000,1.000000
25%,0.991667,43.602634,2.000000,2.000000
50%,2.491667,104.551390,4.000000,4.000000
75%,4.400000,285.365854,9.000000,9.000000
max,14.000000,10500.000000,82.000000,82.000000


In [65]:
nhl.isnull().sum()

Season               0
Team                 0
Position             0
Player               0
Injury Type          0
Cap Hit           6832
Chip              6832
Games Missed         0
Games Missed.1       0
dtype: int64

In [66]:
nhl.shape

(24498, 9)

In [67]:
print("NHL Dataset Duplicates:", nhl.duplicated().sum())
print("\nUnique values per column:\n", nhl.nunique())

NHL Dataset Duplicates: 0

Unique values per column:
 Season              49
Team                35
Position             9
Player            3043
Injury Type         97
Cap Hit            942
Chip              3789
Games Missed        82
Games Missed.1      82
dtype: int64


In [68]:
nhl.columns.value_counts()

Season            1
Team              1
Position          1
Player            1
Injury Type       1
Cap Hit           1
Chip              1
Games Missed      1
Games Missed.1    1
Name: count, dtype: int64

**NHL Dataset Observations:**
*   **Description:** The NHL injuries dataset tracks hockey players' time injured or put on the injured reserve. It typically organizes injuries by body parts and records the number of regular season or playoff games missed due to those specific health instances across seasons.
*   **Data Source:** (https://public.tableau.com.com/app/profile/lw3h/viz/NHLinjurydatabase/NHLInjuryDatabase)
*   **Limitations:** Includes potentially duplicate tracks for missed games making aggregation somewhat uncertain. Lacks highly granular descriptions mapping to exact anatomic locations or severity levels.
*   **Columns:** The dataset contains 9 columns, capturing identifying details like `Season`, `Team`, and `Player`, the descriptive `Injury Type`, and duration counts such as `Games Missed` and the identical `Games Missed.1`.

**Additional Insights:**
*   **Duplicates:** 0 duplicate rows found.
*   **Unique Values:** `Injury Type` has 97 unique categories, which is relatively small and manageable without heavy text parsing. The `Games Missed` and `Games Missed.1` columns seem identical, with 82 unique values each.
* **Injury type column?** `Injury Type`
* **Games missed/recovery/duration?** `Games Missed` and `Games Missed.1`
* **How are injuries described?** Categorical or short text in `Injury Type`.
* **Season/year?** Uses the `Season` column (e.g., "2000/01").
* **Cleanest/most chaotic?** Somewhat clean, but contains potentially duplicate or confusing columns (e.g., `Games Missed.1`). Size: 24,498 rows, 9 columns.

## MLB Injuries

In [69]:
mlb.head()

,date,effectiveDate,notes,player_id,team_id,year,team,team_name,name,dtd,...,cum_season_days,prev_season_days,next_game_season_days,dt,in_season_duration,date_duration,duration,t,position,__index_level_0__
0,2012-04-07 00:00:00.000000,\N,\N,110029,\N,2012,LAA,\N,"Abreu, Bobby",True,...,11,11,11,0,0,0,0,0,batter,0
1,2014-09-28 00:00:00.000000,\N,\N,110029,\N,2014,NYM,\N,"Abreu, Bobby",True,...,565,10,565,264,0,0,0,264,batter,1
2,2012-03-29 00:00:00.000000,\N,\N,112526,\N,2012,OAK,\N,Bartolo Colon,True,...,2,2,2,0,0,0,0,0,pitcher,2
3,2012-06-23 00:00:00.000000,2012-06-18,oakland athletics placed rhp bartolo colon on ...,112526,133,2012,OAK,Athletics,Bartolo Colon,False,...,88,1,98,87,10,10,15,87,pitcher,3
4,2013-08-17 00:00:00.000000,2013-08-14,oakland athletics placed rhp bartolo colon on ...,112526,133,2013,OAK,Athletics,Bartolo Colon,False,...,330,97,342,180,12,12,15,267,pitcher,4


In [70]:
mlb.describe()

,player_id,year,game,injury_span_id_orig,non_mlb_days,il_days_sum,prev_non_mlb_days,cum_season_days,prev_season_days,dt,duration,t,__index_level_0__
count,9626.000000,9626.000000,9626.000000,9626.000000,9626.000000,9626.000000,9626.000000,9626.000000,9626.000000,9626.000000,9626.000000,9626.000000,9626.000000
mean,526727.848639,2017.603678,0.234469,492.297216,15.566071,7.319655,17.653542,1100.109287,976.340017,111.706836,19.886246,582.675774,4812.500000
std,90307.879810,3.093639,0.423689,462.649540,71.727435,14.013965,68.364310,550.884836,572.444094,160.578585,43.903108,429.888121,2778.931179
min,110029.000000,2012.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,457915.000000,2015.000000,0.000000,146.000000,0.000000,0.000000,0.000000,717.250000,572.000000,11.000000,1.000000,223.000000,2406.250000
50%,537953.000000,2018.000000,0.000000,318.000000,0.000000,0.000000,0.000000,1148.000000,1025.000000,51.000000,4.000000,551.000000,4812.500000
75%,595751.000000,2021.000000,0.000000,697.000000,0.000000,10.000000,0.000000,1563.000000,1483.000000,142.000000,20.000000,886.000000,7218.750000
max,683734.000000,2022.000000,1.000000,1500.000000,1243.000000,260.000000,1243.000000,1905.000000,1904.000000,1476.000000,974.000000,1853.000000,9625.000000


In [71]:
mlb.info()

<class 'pandas.DataFrame'>
RangeIndex: 9626 entries, 0 to 9625
Data columns (total 48 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   date                   9626 non-null   str  
 1   effectiveDate          9626 non-null   str  
 2   notes                  9626 non-null   str  
 3   player_id              9626 non-null   int64
 4   team_id                9626 non-null   str  
 5   year                   9626 non-null   int64
 6   team                   9626 non-null   str  
 7   team_name              9626 non-null   str  
 8   name                   9626 non-null   str  
 9   dtd                    9626 non-null   bool 
 10  activated              9626 non-null   str  
 11  il_days                9626 non-null   str  
 12  transfer               9626 non-null   str  
 13  injury_notes           9232 non-null   str  
 14  injury_type_notes      9232 non-null   str  
 15  injury_type            9626 non-null   str  
 16 

In [72]:
mlb.isnull().sum()

date                       0
effectiveDate              0
notes                      0
player_id                  0
team_id                    0
year                       0
team                       0
team_name                  0
name                       0
dtd                        0
activated                  0
il_days                    0
transfer                   0
injury_notes             394
injury_type_notes        394
injury_type                0
injury_location_notes    394
injury_location            0
game                       0
injury_id                  0
injury_date                0
game_date                  0
game_type                  0
prev_is_game               0
next_is_game               0
prev_game_date             0
next_game_date             0
player_first_row           0
player_last_row            0
injury_span_id             0
injury_span_id_orig        0
time_since_last_game       0
non_mlb_days               0
il_days_max                0
il_days_sum   

In [73]:
mlb.shape

(9626, 48)

In [74]:
print("MLB Dataset Duplicates:", mlb.duplicated().sum())
print("\nUnique values per column:\n", mlb.nunique().head(10), "\n... (omitted remainder)")

MLB Dataset Duplicates: 0

Unique values per column:
 date             1846
effectiveDate    1501
notes            3978
player_id        1267
team_id            31
year               11
team               30
team_name          33
name             1265
dtd                 2
dtype: int64 
... (omitted remainder)


In [75]:
mlb.columns.value_counts()

date                     1
effectiveDate            1
notes                    1
player_id                1
team_id                  1
year                     1
team                     1
team_name                1
name                     1
dtd                      1
activated                1
il_days                  1
transfer                 1
injury_notes             1
injury_type_notes        1
injury_type              1
injury_location_notes    1
injury_location          1
game                     1
injury_id                1
injury_date              1
game_date                1
game_type                1
prev_is_game             1
next_is_game             1
prev_game_date           1
next_game_date           1
player_first_row         1
player_last_row          1
injury_span_id           1
injury_span_id_orig      1
time_since_last_game     1
non_mlb_days             1
il_days_max              1
il_days_sum              1
prev_non_mlb_days        1
start_date               1
p

**MLB Dataset Observations:**
*   **Description:** The MLB injury database records baseball injuries at a high level of detail, registering stints on the Injured List (IL) and tracking granular anatomical data, precise dates of time missed, and detailed locations of injuries occurring throughout seasons.
*   **Data Source:** https://github.com/ssharpe42/mlb-injury/tree/master/data
*   **Limitations:** High dimensionality (48 columns) could introduce significant complexity and noise making feature selection crucial. The excessive null values across its sparse parameters may require aggressive imputation strategies.
*   **Columns:** It is the most extensive dataset with 48 columns. Key fields include explicit attributes for `injury_type` and `injury_location`, explicit duration measures (`il_days`, `in_season_duration`), temporal context (`year`, `date`), and general text in `notes`.

**Additional Insights:**
*   **Duplicates:** 0 duplicate rows found.
*   **Unique Values:** Shows 11 unique years, aligning with a likely 2010-2020 layout. Contains deep granularity across multiple text fields (e.g. nearly 4,000 unique `notes`), emphasizing the rich but heavy memory profile of this dataset.
* **Injury type column?** Multiple options: `injury_type`, `injury_type_notes`, `injury_location`
* **Games missed/recovery/duration?** Explicit duration tracking in multiple columns: `il_days`, `in_season_duration`, `date_duration`, `duration`
* **How are injuries described?** Extensively broken down into structured details (`injury_type`, `injury_location`, `injury_notes`).
* **Season/year?** Has both `year` and `date`.
* **Cleanest/most chaotic?** Most detailed and feature-rich (48 columns). Might require dropping redundant columns but very comprehensively structured. Size: 9,626 rows, 48 columns.

In [76]:
# Put all four datasets in one dictionary so we can loop through them together.
datasets = {
    "NFL": nfl,
    "NBA": nba,
    "NHL": nhl,
    "MLB": mlb,
}

# Save only the column indexes, because pandas Index supports intersection() and union().
column_indexes = {name: df.columns for name, df in datasets.items()}

# Start from NFL columns, then keep only the column names that also exist in the other datasets.
common_to_all = column_indexes["NFL"]
for columns in column_indexes.values():
    common_to_all = common_to_all.intersection(columns)

# Compare every pair of datasets to find exact column-name matches between them.
pairwise_intersections = {
    f"{left} & {right}": column_indexes[left].intersection(column_indexes[right]).tolist()
    for left, right in combinations(column_indexes, 2)
}

# Build one combined list with every unique column name that appears in any dataset.
all_columns = pd.Index([])
for columns in column_indexes.values():
    all_columns = all_columns.union(columns)

print("Common columns across all four datasets:")
print(common_to_all.tolist())

print("\nExact pairwise column-name intersections:")
for pair, columns in pairwise_intersections.items():
    print(f"{pair}: {columns}")

print(f"\nUnion of all column names ({len(all_columns)} columns):")
print(all_columns.tolist())


Common columns across all four datasets:
[]

Exact pairwise column-name intersections:
NFL & NBA: []
NFL & NHL: ['Player']
NFL & MLB: ['year', 'team']
NBA & NHL: ['Team']
NBA & MLB: []
NHL & MLB: []

Union of all column names (69 columns):
['Acquired', 'Cap Hit', 'Chip', 'Date', 'Games Missed', 'Games Missed.1', 'Injury Type', 'Notes', 'Player', 'Player_id', 'Position', 'Relinquished', 'Season', 'Team', 'Unnamed: 0', '__index_level_0__', 'activated', 'cum_season_days', 'date', 'date_duration', 'dt', 'dtd', 'duration', 'earliest_injury', 'effectiveDate', 'game', 'game_date', 'game_type', 'games_in_season', 'il_days', 'il_days_max', 'il_days_sum', 'in_season_duration', 'injury_date', 'injury_id', 'injury_location', 'injury_location_notes', 'injury_notes', 'injury_span_id', 'injury_span_id_orig', 'injury_type', 'injury_type_notes', 'injury_types', 'latest_injury', 'name', 'next_game_date', 'next_game_season_days', 'next_is_game', 'non_mlb_days', 'notes', 'num_games_injured', 'num_games_mi

### Dataset Comparison
*   **Largest Dataset:** **NBA** leads in total records (27,105 rows), followed by **NHL** (24,498 rows) and **NFL** (17,389 rows). In terms of features (columns), **MLB** is the absolute leader with 48 columns per record (though it only has 9,626 rows).
*   **Cleanest (Structured):** **MLB** and **NFL**. MLB offers the richest and most detailed structure for analysis due to multiple columns for injury duration and location. NFL is also very well-structured and straightforward to work with.
*   **Most Chaotic:** Unquestionably **NBA**. The lack of categorical columns means that serious text parsing and NLP will be required on the `Notes` column to extract the injury types and missed time. NHL also has minor issues (such as the duplicate/unclear `Games Missed.1` column), but it is much better structured than NBA.
*   **Column-name overlap:** Using pandas-style `Index.intersection()` and `Index.union()` logic, there are no exact column names shared by all four datasets because naming conventions differ by case and wording. Exact pairwise overlaps are limited: **NFL** and **MLB** share `year` and `team`; **NBA** and **NHL** share `Team`; and **NFL** and **NHL** share `Player`. The strongest derivative or semantic matches are player identifiers (`Player_id`/`player_id`, `Player`/`name`, `Relinquished`), team fields (`team`, `Team`, `team_id`, `team_name`), season/date fields (`year`, `Season`, `Date`, `date`, `injury_date`, `game_date`, `start_date`), injury descriptions (`injury_types`, `Injury Type`, `injury_type`, `injury_location`, `Notes`, `notes`), and duration fields (`num_games_missing`, `num_games_injured`, `Games Missed`, `Games Missed.1`, `il_days`, `duration`, `in_season_duration`). Different or dataset-specific names include NHL financial columns (`Cap Hit`, `Chip`), MLB event/status columns (`dtd`, `activated`, `transfer`, game-date context fields), NBA transaction columns (`Acquired`, `Relinquished`), and NFL season aggregate fields (`games_in_season`, `earliest_injury`, `latest_injury`). The union of columns is therefore much larger than the intersection, so standardization should map derivative names into common fields before merging.

### Preliminary Conclusion
The datasets exhibit substantial heterogeneity in structure, granularity, and injury reporting conventions.

The NFL and MLB datasets are more structured and suitable for direct analysis, while the NBA dataset requires significant preprocessing due to its reliance on unstructured text.

These differences necessitate a robust standardization pipeline before any meaningful cross-league comparison can be performed.

The datasets originate from different sources and may use varying definitions and reporting standards for injuries, which should be considered when interpreting the results.

# 2. Data Cleaning

 After text normalization, some values may become empty strings if the original field contained only punctuation, numbers, or irregular symbols. Replacing empty strings with `NaN` makes missing values explicit and prevents blank categories from being treated as valid injury labels.


## 2.1 NFL Cleaning
### Text Normalization

The NFL dataset stores injury information in the `injury_types` field, where similar injuries can appear with different capitalization, separators, or wording.  
Before this field can be compared with the other leagues, these labels need a common textual format.

This step converts the NFL injury labels to lowercase, removes punctuation, and trims extra whitespace.  
The purpose is to reduce artificial variation in the data, so entries such as "Knee", "knee injury", and "knee-injury" are not treated as separate injury categories.

This operation preserves the medical meaning of the original NFL records while making the categories more consistent for later aggregation and cross-league analysis.


In [77]:
nfl["injury_types"] = nfl["injury_types"].apply(clean_text)

**Reasoning:** Applying the `clean_text` function normalizes the text by converting it to lowercase and removing special characters. This standardizes the descriptions so that identical injuries (e.g., "Knee" and "knee-injury") are grouped correctly for analysis.

In [78]:
nfl = normalize_empty_strings(nfl, ["injury_types"])
nfl = nfl.dropna(subset=["injury_types"])

**Reasoning:** Observations without a recorded injury type provide no value for learning about specific injury distributions or impacts. Dropping these missing values (`NaN`) ensures our dataset strictly contains actionable and relevant injury events.

In [79]:
nfl["num_games_missing"] = pd.to_numeric(nfl["num_games_missing"], errors="coerce")

**Reasoning:** The `num_games_missing` column must be a formal numeric format for future statistical aggregations (like averages or totals). Using `errors="coerce"` safely handles any unexpected non-numeric string entries by cleanly replacing them with `NaN`.

In [80]:
final_df["injury_type"].nunique() if "injury_type" in df.columns else "N/A"

9

**Reasoning:** Counting the unique standardized injury types provides a quick sanity check. It allows us to immediately see if our normalization steps successfully consolidated the disparate text descriptions into a manageable number of distinct categories.

In [81]:
nfl = nfl.rename(columns={
    "injury_types": "injury_type",
    "num_games_missing": "games_missed"
})

**Reasoning:** Renaming `injury_types` to `injury_type` and `num_games_missing` to `games_missed` standardizes our column headers. This creates consistency across the NFL, NHL, and MLB datasets, enabling seamless merging and cross-league comparative analysis.

## 2.2 NBA Cleaning
### Text Normalization

The NBA dataset does not provide a dedicated injury-type column; the relevant information is embedded in the free-text `Notes` field.  
Because these notes combine injury details with transaction-style wording, they must be normalized before keyword extraction.

This step converts the NBA notes to lowercase, removes punctuation, and trims extra whitespace.  
The purpose is to create a cleaner text baseline so that the later parsing function can detect injury terms consistently across differently written reports.

This operation does not reinterpret the note itself; it only standardizes the wording so injury categories can be derived more reliably.

In [82]:
nba["Notes"] = nba["Notes"].apply(clean_text)
nba = normalize_empty_strings(nba, ["Notes"])
nba = nba.dropna(subset=["Notes"])


**Reasoning:** The `Notes` column contains unstructured text. Converting it to strings and applying `clean_text` ensures a uniform baseline (lowercase, no punctuation) before we attempt to parse specific injury categories from it.

In [83]:
nba["injury_type"] = nba["Notes"].apply(extract_nba_injury_type)

**Reasoning:** Unlike the other datasets, the NBA data lacks a direct injury type column. Using our custom `extract_nba_injury_type` function scans the cleaned notes for keywords to conditionally categorize each log into standard injury groups.

In [84]:
nba["Date"] = pd.to_datetime(nba["Date"], errors="coerce")
nba["year"] = nba["Date"].dt.year


**Reasoning:** Converting the original `Date` string to a formal Pandas datetime format ensures we can perform reliable temporal operations and time-series aggregations later without running into string parsing errors.

In [85]:
nba["year"] = nba["Date"].dt.year

**Reasoning:** Extracting the `year` from our newly created datetime objects creates a designated column that directly matches the structuring format used in both the NFL and MLB datasets, making merging easier.

In [86]:
nba = nba[nba["injury_type"] != "unknown"]


**Reasoning:** Some NBA notes may describe rest days or records without a detectable injury category. Removing `unknown` values, and optionally `rest`, keeps the dataset focused on actual injury-related observations.


## 2.3 NHL Cleaning
### Text Normalization

The NHL dataset contains a direct `Injury Type` field, but its values still need to align with the naming conventions used in the other cleaned datasets.  
Normalizing this column helps avoid separate categories caused only by formatting differences.

This step converts NHL injury descriptions to lowercase, removes punctuation, and trims extra whitespace after the column is renamed to `injury_type`.  
The purpose is to keep body-part and injury labels comparable with the standardized NFL, NBA, and MLB outputs.

This operation does not change the recorded medical meaning; it only prepares the NHL labels for consistent grouping and merging.

In [87]:
nhl = nhl.drop(columns=["Games Missed.1"])

**Reasoning:** The `Games Missed.1` column appears to be a redundant duplicate of the `Games Missed` column. Dropping it reduces clutter, saves memory, and removes potential confusion during analysis.

In [88]:
nhl = nhl.rename(columns={
    "Injury Type": "injury_type",
    "Games Missed": "games_missed",
    "Season": "season"
})

**Reasoning:** Renaming columns to standard snake_case formatting with descriptive names makes them easier to reference programmatically. It also aligns the NHL dataset's structure with the other sports datasets.

In [89]:
nhl["injury_type"] = nhl["injury_type"].apply(clean_text)

**Reasoning:** Applying `clean_text` normalizes the injury descriptions by converting them to lowercase and stripping special characters. This ensures that identical injuries (e.g., "Knee" and "knee injury") are consistently grouped and parsed.

In [90]:
nhl = nhl.drop(columns=["Cap Hit", "Chip"])


**Reasoning:** `Cap Hit` and `Chip` contain many missing values and are not required for injury-type or games-missed analysis. Dropping them reduces noise and keeps the NHL dataset aligned with the shared analytical schema.


## 2.4 MLB Cleaning
### Text Normalization

The MLB dataset is richer and more structured than the others, with separate fields for `injury_type`, `injury_location`, duration, and notes.  
Even with this structure, the textual injury fields can contain inconsistent capitalization, punctuation, or spacing.

This step normalizes the MLB injury labels before they are used alongside the injury-location information.  
The purpose is to prevent equivalent baseball injury categories from being split because of superficial text differences.

This operation preserves the original meaning of the MLB injury records while improving consistency for summary statistics and cross-league comparison.

In [91]:
mlb = mlb[[
    "year",
    "injury_type",
    "injury_location",
    "il_days",
    "notes"
]]

**Reasoning:** The MLB dataset has 48 columns, many of which contain sparse or irrelevant data. Subsetting down to these 5 core columns trims the noise and focuses our analysis strictly on the critical injury and duration attributes we need.

In [92]:
mlb = mlb.rename(columns={
    "il_days": "games_missed"
})

**Reasoning:** Renaming `il_days` (injured list days) to `games_missed` standardizes the duration metric's column name. This consistency allows for easier merging and cross-league comparison with the matching NHL and NFL metrics.

In [93]:
mlb["injury_type"] = mlb["injury_type"].apply(clean_text)

**Reasoning:** Applying the `clean_text` function to the `injury_type` column removes special characters and converts the text to lowercase. This prevents duplicate categorical groupings caused by arbitrary text formatting differences.

In [94]:
mlb["injury_location"] = mlb["injury_location"].apply(clean_text)


**Reasoning:** Similar to the injury type, processing `injury_location` through `clean_text` normalizes the text entries. This ensures anatomical locations are uniformly structured for accurate aggregation and comparative analysis.

In [95]:
mlb = mlb.drop(columns=["notes"])


**Reasoning:** If the analysis relies only on structured injury fields, the free-text `notes` column can be removed to reduce dimensional noise and keep the cleaned dataset focused on comparable variables.


## 2.5 Post-cleaning Validation

### Post-cleaning Validation

After applying cleaning and preprocessing steps, it is necessary to validate the integrity of the datasets.

This step evaluates:
- the presence of missing values
- the overall dataset size after cleaning

The goal is to ensure that the preprocessing steps have not introduced data loss or inconsistencies that could affect subsequent analysis.

In [96]:
nfl["league"] = "NFL"
nba["league"] = "NBA"
nhl["league"] = "NHL"
mlb["league"] = "MLB"

In [97]:
nfl = nfl.drop_duplicates()
nba = nba.drop_duplicates()
nhl = nhl.drop_duplicates()
mlb = mlb.drop_duplicates()


**Reasoning:** Duplicate rows can overrepresent certain injury events and bias frequency-based summaries. Removing duplicates after the main fields have been standardized helps ensure that repeated records do not inflate injury counts.


In [98]:
for df, name in zip([nfl, nba, nhl, mlb], ["NFL", "NBA", "NHL", "MLB"]):
    print(f"\n{name}")
    print(df.isnull().sum())
    print(df.shape)


NFL
Unnamed: 0           0
Player_id            0
Player               0
year                 0
team                 0
games_in_season      0
games_missed         0
num_games_injured    0
injury_type          0
earliest_injury      0
latest_injury        0
league               0
dtype: int64
(15910, 12)

NBA
Date                0
Team                2
Acquired        17544
Relinquished     9544
Notes               0
injury_type         0
year                0
league              0
dtype: int64
(27085, 8)

NHL
season          0
Team            0
Position        0
Player          0
injury_type     0
games_missed    0
league          0
dtype: int64
(24498, 7)

MLB
year               0
injury_type        0
injury_location    0
games_missed       0
league             0
dtype: int64
(1285, 5)


In [99]:
for df, name in zip([nfl, nba, nhl, mlb], ["NFL", "NBA", "NHL", "MLB"]):
    print(f"\n{name}")
    print("Unique injury types:", df["injury_type"].nunique() if "injury_type" in df.columns else "N/A")


NFL
Unique injury types: 2980

NBA
Unique injury types: 20

NHL
Unique injury types: 97

MLB
Unique injury types: 9


### Column Standardization

Different datasets use varying naming conventions for similar concepts.  
For example, the NFL dataset uses the column name "injury_types", while other datasets use variations such as "Injury Type" or "injury_type".

To ensure consistency across datasets and enable seamless merging, column names are standardized to a common schema.

This step reduces semantic ambiguity and allows unified processing in subsequent stages.

# 3. Injury Standartization and Categorization

### Injury Categorization

Injury descriptions across datasets are highly heterogeneous and cannot be directly compared.

To enable cross-league analysis, injury descriptions are mapped into broader categories:
- Head
- Upper Body
- Lower Body
- Illness
- Rest/Non-injury
- Unknown
- Other

The anatomical categories preserve medically relevant body-region groupings, while the additional non-anatomical categories prevent illness, rest, and undisclosed records from inflating the generic `Other` group.

Special attention is given to head injuries due to their long-term neurological implications.

## 3.1 NFL

In [100]:
nfl["injury_category"] = nfl["injury_type"].apply(categorize_injury)

**Reasoning:** The NFL dataset has thousands of unique injury types, which creates excessive dimensionality for analysis. By applying `categorize_injury`, we map these granular descriptions into broad anatomical categories (Head, Upper Body, Lower Body, Other). This reduces noise, enables meaningful cross-league comparisons, and highlights medically relevant injury patterns.

## 3.2 NBA

In [101]:
if "Notes" in nba.columns:
    nba["injury_context"] = (
        nba["injury_type"].fillna("") + " " +
        nba["Notes"].fillna("")
    ).str.strip()
else:
    nba["injury_context"] = nba["injury_type"].fillna("")

nba["injury_category"] = nba["injury_context"].apply(categorize_injury)

**Reasoning:** The NBA dataset relies on free-text notes, and the extracted `injury_type` can collapse many unrecognized descriptions into `other`. When `Notes` is still available, combining it with `injury_type` preserves the original context before categorization. If this cell is rerun after schema harmonization and `Notes` has already been removed, the code falls back to `injury_type` so the notebook remains executable.

## 3.3 NHL

In [102]:
nhl["injury_category"] = nhl["injury_type"].apply(categorize_injury)

**Reasoning:** The NHL dataset contains 97 unique standardized injury types organized by body part. While this is more manageable than the NFL's thousands, consolidating them into four broad anatomical categories (Head, Upper Body, Lower Body, Other) ensures direct comparability with all other leagues and simplifies pattern identification across sports.

## 3.4 MLB

In [103]:
if "injury_location" in mlb.columns:
    mlb["injury_combined"] = (
        mlb["injury_type"].fillna("") + " " +
        mlb["injury_location"].fillna("")
    ).str.strip()
    mlb["injury_category"] = mlb["injury_combined"].apply(categorize_injury)
else:
    mlb["injury_category"] = mlb["injury_type"].apply(categorize_injury)

**Reasoning:** The MLB dataset contains both `injury_type` and `injury_location`, so combining them gives the categorization function more anatomical context than `injury_type` alone. This is especially important for generic injury labels such as strains, sprains, fractures, or inflammation, where the body location determines whether the record belongs to the upper body, lower body, head, or another group.

## 2.5 Result Validation

In [104]:
for df, name in zip([nfl, nba, nhl, mlb], ["NFL", "NBA", "NHL", "MLB"]):
    print(f"\n{name}")
    print(df["injury_category"].value_counts())


NFL
injury_category
Lower Body         10060
Upper Body          2460
Head                1559
Illness             1137
Other                339
Unknown              285
Rest/Non-injury       70
Name: count, dtype: int64

NBA
injury_category
Other              13309
Lower Body          5629
Rest/Non-injury     5200
Upper Body          1845
Illness              758
Head                 336
Unknown                8
Name: count, dtype: int64

NHL
injury_category
Lower Body    9327
Upper Body    7915
Illness       3031
Head          2004
Unknown       1711
Other          510
Name: count, dtype: int64

MLB
injury_category
Upper Body    476
Lower Body    378
Unknown       285
Other          97
Head           49
Name: count, dtype: int64


**Reasoning:** This validation displays the distribution of injury categories for each league after categorization. The `value_counts()` output confirms that the mapping executed successfully and shows whether any league still has a high share of `Other`, `Unknown`, illness-related, or rest-related records that may require additional review before comparative analysis.

In [105]:
print(nba["injury_type"].value_counts().head(20))
print("\n---\n")
print(nhl["injury_type"].value_counts().head(20))
print("\n---\n")
print(mlb["injury_type"].value_counts().head(20))

injury_type
other        13536
knee          2730
ankle         2187
foot          1022
illness        974
back           940
rest           860
lower leg      716
hamstring      649
hand           537
shoulder       530
head           467
groin          431
hip            430
quad           330
wrist          255
arm            191
core           108
neck            96
chest           96
Name: count, dtype: int64

---

injury_type
upper body       3576
lower body       3575
undisclosed      1711
knee             1708
illness          1549
groin            1294
shoulder         1205
concussion       1121
back              900
flu               814
ankle             753
foot              658
covid related     643
leg               565
hand              557
hip               497
wrist             351
head              346
neck              252
finger            235
Name: count, dtype: int64

---

injury_type
tear strain sprain    311
misc unk              249
minor                 212
in

**Reasoning:** This cell examines the top 20 most frequent injury types for each league before final categorization. By comparing these rankings, we gain insight into the dominant injury patterns in each sport: (1) NBA's extracted categories reveal which injury keywords are most prevalent in the free-text notes, (2) NHL's view shows the distribution of its 97 standardized injury types, and (3) MLB's top injuries highlight the most common specific injuries and anatomical locations recorded in the structured data. This exploratory view validates that text normalization and extraction functions worked correctly, confirms that the data is suitable for categorization, and provides context for understanding how broad anatomical categories will consolidate these granular observations.

In [106]:
for df, name in zip([nfl, nba, nhl, mlb], ["NFL", "NBA", "NHL", "MLB"]):
    print(f"\n{name} - remaining Other injury types")
    print(
        df.loc[df["injury_category"] == "Other", "injury_type"]
        .value_counts()
        .head(15)
    )


NFL - remaining Other injury types
injury_type
quadricep           99
related             35
served              34
notinjuryrelated    24
bicep               18
collarbone          10
tricep               9
stinger              7
hernia               6
disciplinary         6
rightshoulder        6
heart                4
kidney               4
rightgroin           4
lung                 3
Name: count, dtype: int64

NBA - remaining Other injury types
injury_type
other    13309
Name: count, dtype: int64

NHL - remaining Other injury types
injury_type
hernia           69
collarbone       47
heart            38
soreness         34
charley horse    33
headache         29
blood clots      29
cheek            27
appendectomy     20
sternum          16
throat           15
kidney           14
dizziness        13
torso            13
mid body         11
Name: count, dtype: int64

MLB - remaining Other injury types
injury_type
tear strain sprain    26
break fracture        17
contusion bruise    

**Reasoning:** Inspecting the most frequent remaining `Other` values helps evaluate the quality of the mapping after the main keyword expansion. If repeated medical terms still appear here, they can be added to the categorization logic; if the values are vague labels such as `misc`, `unknown`, or generic statuses, they should remain separate from anatomical injury categories.

In [107]:
for df, name in zip([nfl, nba, nhl, mlb], ["NFL", "NBA", "NHL", "MLB"]):
    print(f"\n{name}")
    print(df["injury_category"].value_counts(normalize=True))


NFL
injury_category
Lower Body         0.632307
Upper Body         0.154620
Head               0.097989
Illness            0.071464
Other              0.021307
Unknown            0.017913
Rest/Non-injury    0.004400
Name: proportion, dtype: float64

NBA
injury_category
Other              0.491379
Lower Body         0.207827
Rest/Non-injury    0.191988
Upper Body         0.068119
Illness            0.027986
Head               0.012405
Unknown            0.000295
Name: proportion, dtype: float64

NHL
injury_category
Lower Body    0.380725
Upper Body    0.323088
Illness       0.123724
Head          0.081803
Unknown       0.069842
Other         0.020818
Name: proportion, dtype: float64

MLB
injury_category
Upper Body    0.370428
Lower Body    0.294163
Unknown       0.221790
Other         0.075486
Head          0.038132
Name: proportion, dtype: float64


### Categorization Quality Assessment

The initial categorization produced a high proportion of `Other` injuries because several different concepts were being mixed together: true unmapped injury terms, illness, rest, personal absences, and undisclosed records.

To address this, the categorization logic was expanded in three ways:
- broader anatomical keyword coverage for head, upper-body, and lower-body injuries
- separate non-anatomical categories for illness, rest/non-injury cases, and unknown or undisclosed records
- richer context for NBA and MLB by combining available text fields before categorization

After these changes, `Other` should represent genuinely unmapped injury text rather than a catch-all bucket for every non-standard record. The remaining `Other` values should be reviewed with the diagnostic cell above before deciding whether to add more keywords.

# 4. Schema harmonization and data integration

### Column Alignment

Before merging datasets from different sources, it is necessary to standardize the schema.

Each dataset originally uses different column names and structures.  
To enable concatenation and comparative analysis, all datasets are reduced to a common set of variables:

- league
- year
- injury_type
- injury_category
- games_missed

This ensures structural consistency across all observations.

## 4.1 NFL

In [108]:
nfl = nfl[["league", "year", "injury_type", "injury_category", "games_missed"]]

**Reasoning:** The NFL dataset already contains the standardized analytical fields needed for integration. Selecting only `league`, `year`, `injury_type`, `injury_category`, and `games_missed` removes dataset-specific columns and aligns NFL with the common schema used for all leagues.

## 4.2 NBA

In [109]:
nba["games_missed"] = None

nba = nba[["league", "year", "injury_type", "injury_category", "games_missed"]]

**Reasoning:** The NBA dataset does not provide a structured games-missed column, so `games_missed` is added as a placeholder to preserve schema consistency. This allows NBA records to be merged with the other datasets while making the missing duration information explicit.

## 4.3 NHL

In [110]:
if "season" in nhl.columns:
    nhl = nhl.rename(columns={"season": "year"})

nhl = nhl[["league", "year", "injury_type", "injury_category", "games_missed"]]

**Reasoning:** The NHL dataset uses `season` rather than `year`, so the column is renamed before alignment. After that, the dataset is reduced to the shared analytical fields required for cross-league comparison.

## 4.4 MLB

In [111]:
mlb = mlb.rename(columns={"il_days": "games_missed"})

mlb = mlb[["league", "year", "injury_type", "injury_category", "games_missed"]]

**Reasoning:** MLB records use `il_days` to represent time missed on the injured list. Renaming it to `games_missed` standardizes the duration field name, and selecting the common columns prepares MLB for integration with the other leagues.

In [112]:
print(nfl.columns)
print(nba.columns)
print(nhl.columns)
print(mlb.columns)

Index(['league', 'year', 'injury_type', 'injury_category', 'games_missed'], dtype='str')
Index(['league', 'year', 'injury_type', 'injury_category', 'games_missed'], dtype='str')
Index(['league', 'year', 'injury_type', 'injury_category', 'games_missed'], dtype='str')
Index(['league', 'year', 'injury_type', 'injury_category', 'games_missed'], dtype='str')


**Reasoning:** Printing the columns for each cleaned dataset verifies that all four datasets now share the same schema before concatenation. This is a quick structural check that helps catch missing, misnamed, or extra columns before merging.

# Merging and Validation

In [113]:
final_df = pd.concat([nfl, nba, nhl, mlb], ignore_index=True)

**Reasoning:** Concatenating the four aligned datasets creates one unified table for cross-league analysis. Using `ignore_index=True` resets the row index so the merged dataset has a clean continuous index independent of the original dataset row numbers.

In [114]:
final_df.head()
final_df.info()

print(final_df["league"].value_counts())
print(final_df["injury_category"].value_counts())

<class 'pandas.DataFrame'>
RangeIndex: 68778 entries, 0 to 68777
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   league           68778 non-null  str   
 1   year             68778 non-null  object
 2   injury_type      68778 non-null  str   
 3   injury_category  68778 non-null  str   
 4   games_missed     41693 non-null  object
dtypes: object(2), str(3)
memory usage: 2.6+ MB
league
NBA    27085
NHL    24498
NFL    15910
MLB     1285
Name: count, dtype: int64
injury_category
Lower Body         25394
Other              14255
Upper Body         12696
Rest/Non-injury     5270
Illness             4926
Head                3948
Unknown             2289
Name: count, dtype: int64


**Reasoning:** This validation cell inspects the merged dataset immediately after concatenation. `head()` previews the resulting structure, `info()` checks data types and missing values, and the value counts confirm the contribution of each league and the distribution of injury categories.

In [115]:
final_df["year"] = pd.to_numeric(final_df["year"], errors="coerce")

**Reasoning:** Converting `year` to a numeric type makes the field suitable for chronological filtering, grouping, and statistical analysis. Invalid or non-convertible values are coerced to missing values so they can be identified instead of silently distorting time-based analysis.

In [116]:
final_df["games_missed"] = pd.to_numeric(final_df["games_missed"], errors="coerce")

**Reasoning:** Converting `games_missed` to numeric format enables quantitative comparisons of injury duration across leagues. Values that cannot be converted are set to missing, which is appropriate for datasets such as NBA where structured missed-game duration is unavailable.

### Data Type Normalization

Certain columns such as `year` and `games_missed` were initially stored as non-numeric types.

To enable quantitative analysis and statistical testing, these columns were converted to numeric format.

Non-convertible values were set to missing (NaN), ensuring that invalid entries do not bias the analysis.

In [117]:
final_df["injury_category"].value_counts(normalize=True)

injury_category
Lower Body         0.369217
Other              0.207261
Upper Body         0.184594
Rest/Non-injury    0.076623
Illness            0.071622
Head               0.057402
Unknown            0.033281
Name: proportion, dtype: float64

**Reasoning:** Normalized value counts show the proportional distribution of injury categories in the final integrated dataset. This provides a compact quality check for the categorization process and highlights whether categories such as `Other`, `Unknown`, or non-injury statuses remain unusually large.

### Final Injury Categorization

After iterative refinement, injury descriptions were successfully mapped into meaningful categories.

The final distribution shows:
- a balanced representation of anatomical injury groups
- a controlled proportion of "Other" (~20%), indicating remaining ambiguity without overfitting
- explicit separation of non-injury statuses such as rest and illness

This ensures that the dataset reflects real injury patterns while preserving data integrity and avoiding artificial classification bias.

In [118]:
final_df.to_csv("../datasets/processed/cleaned_injuries.csv", index=False)

## References

- Project raw datasets: MLB, NBA, NFL, and NHL injury files stored in `datasets/raw/`.
- McKinney, W. (2010). Data structures for statistical computing in Python. *Proceedings of the 9th Python in Science Conference*. https://conference.scipy.org/proceedings/scipy2010/mckinney.html
- The pandas development team. pandas documentation. https://pandas.pydata.org/docs/
- Harris, C. R., Millman, K. J., van der Walt, S. J., et al. (2020). Array programming with NumPy. *Nature*, 585, 357-362. https://doi.org/10.1038/s41586-020-2649-2
- Python Software Foundation. `re` - Regular expression operations. https://docs.python.org/3/library/re.html
- Python Software Foundation. `itertools` - Functions creating iterators for efficient looping. https://docs.python.org/3/library/itertools.html